In [71]:
import pandas as pd
import numpy as np
import psycopg
from psycopg import sql
import os
import sys
from dotenv import load_dotenv
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"

In [ ]:
## Top hero winrates
columns=['id', 'didRadiantWin', 'gameVersionId', 'heroId', 'isRadiant', 'isVictory', 'name']
query = '''
    SELECT md.{0}, md.{1}, md.{2}, mp.{3}, mp.{4}, mp.{5}, heroes.{6} 
    FROM match_details md
    INNER JOIN match_players mp
    ON md.id = mp.match_id
    INNER JOIN heroes
    ON mp."heroId" = heroes.id;
'''
results = query_identifier_select(conn_str, query, identifiers=columns)
df = pd.DataFrame(results, columns=columns)
hero_stats = df.groupby(['name'])['isVictory'].aggregate(['mean', 'count'])
hero_stats.columns = ['Win rate', 'Games played']
hero_stats['Win rate'] = (hero_stats['Win rate']*100).round(2)
hero_stats = hero_stats.sort_values(by='Win rate', ascending=False)
hero_stats = hero_stats[hero_stats['Games played'] >= 200]
hero_stats

,id,didRadiantWin,gameVersionId,heroId,isRadiant,isVictory,name
0,8183642521,True,179,9,True,True,npc_dota_hero_mirana
1,8183642521,True,179,29,True,True,npc_dota_hero_tidehunter
2,8183642521,True,179,106,True,True,npc_dota_hero_ember_spirit
3,8183642521,True,179,63,True,True,npc_dota_hero_weaver
4,8183642521,True,179,19,True,True,npc_dota_hero_tiny


In [181]:
## Pick and ban rates
## IMPORTANT: matches were dropped that had no pick-ban phase and where a pick/ban was missing
query = '''
    SELECT mpb.*, heroes.name, heroes.localized_name
    FROM match_pick_bans mpb
    INNER JOIN heroes
    ON mpb."heroId" = heroes.id;
'''
results = query_identifier_select(conn_str, query)
df = pd.DataFrame(results, columns=['id', 'matchId', 'isPick', 'heroId', 'order', 'isRadiant', 'heroName', 'displayName'])
orders = df.groupby('matchId')['order'].max().sort_values(ascending=True)
matches_to_drop = orders.loc[lambda x: x == 23].index
df = df[df['matchId'].isin(matches_to_drop)]

In [183]:
first_bans = df[df['order'] <= 6]
first_bans = first_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_ban_rates = first_bans / len(df['matchId'].unique())

In [184]:
first_picks = df[(df['order'] >= 7) & (df['order'] <= 8)] 
first_picks = first_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_pick_rates = first_picks / len(df['matchId'].unique())

In [185]:
second_bans = df[(df['order'] >= 9) & (df['order'] <= 11)]
second_bans = second_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_ban_rates = second_bans / len(df['matchId'].unique())

In [ ]:
second_picks = df[(df['order'] >= 12) & (df['order'] <= 17)] 
second_picks = second_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_pick_rates = second_picks / len(df['matchId'].unique())

In [187]:
third_bans = df[(df['order'] >= 18) & (df['order'] <= 21)]
third_bans = third_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_ban_rates = third_bans / len(df['matchId'].unique())

In [188]:
third_picks = df[(df['order'] >= 22)] 
third_picks = third_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_pick_rates = third_picks / len(df['matchId'].unique())

In [189]:
overall_pbs = df.groupby('displayName')['displayName'].count().sort_values(ascending=False)
overal_pbs_rates = overall_pbs / len(df['matchId'].unique())

In [190]:
overal_pbs_rates

displayName
Beastmaster         0.669125
Puck                0.645323
Nature's Prophet    0.602749
Ursa                0.519611
Templar Assassin    0.507543
                      ...   
Spectre             0.011063
Legion Commander    0.009722
Lich                0.008046
Arc Warden          0.006369
Treant Protector    0.006034
Name: displayName, Length: 126, dtype: float64